<a href="https://colab.research.google.com/github/qusaiAboSondos/spoken/blob/claude%2Fproject-step-by-step-jx5y1a/notebooks/Spoken_Deepfake_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spoken — AI-generated speech detection (run on Colab)

Run these cells **in order**, top to bottom (Shift+Enter on each). Each cell has a comment explaining what it does.

At the end you'll download a `spoken_results.zip` file — send that back in the chat and Claude will continue from there (generalization experiment + report).

In [2]:
# 1) Clone the project repo and switch to the working branch
!git clone https://github.com/qusaiAboSondos/spoken.git
%cd spoken
!git checkout claude/project-step-by-step-jx5y1a

Cloning into 'spoken'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 32 (delta 7), reused 30 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 21.50 KiB | 21.50 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/spoken
Already on 'claude/project-step-by-step-jx5y1a'
Your branch is up to date with 'origin/claude/project-step-by-step-jx5y1a'.


In [9]:
from google.colab import files
files.upload()   # اختار kaggle.json من جهازك لما يفتح لك مربع الرفع

{}

In [12]:
!cp data/raw/LA/ASVspoof2019_LA_cm_protocols/*.txt data/protocols/
!ls data/protocols/

ASVspoof2019.LA.cm.dev.trl.txt	  dummy_dev_protocol.txt
ASVspoof2019.LA.cm.eval.trl.txt   dummy_protocol.txt
ASVspoof2019.LA.cm.train.trn.txt


In [13]:
!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.train.trn.txt \
  --n-per-class 1500 --output data/protocols/train_subset.txt

!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.dev.trl.txt \
  --n-per-class 500 --output data/protocols/dev_subset.txt

bonafide: 1500/2580 kept
spoof: 1500/22800 kept
Wrote 3000 lines -> data/protocols/train_subset.txt
bonafide: 500/2548 kept
spoof: 500/22296 kept
Wrote 1000 lines -> data/protocols/dev_subset.txt


In [14]:
!python -m src.extract_features \
  --protocol data/protocols/train_subset.txt \
  --audio-dir data/raw/LA/ASVspoof2019_LA_train/flac \
  --features mfcc lfcc spectral --output data/features/train.npz

!python -m src.extract_features \
  --protocol data/protocols/dev_subset.txt \
  --audio-dir data/raw/LA/ASVspoof2019_LA_dev/flac \
  --features mfcc lfcc spectral --output data/features/dev.npz

extracting features: 100% 3000/3000 [01:32<00:00, 32.49it/s]
Saved 3000 utterances -> data/features/train.npz
  mfcc: dim=120
  lfcc: dim=120
  spectral: dim=12
extracting features: 100% 1000/1000 [00:29<00:00, 33.96it/s]
Saved 1000 utterances -> data/features/dev.npz
  mfcc: dim=120
  lfcc: dim=120
  spectral: dim=12


In [16]:
!find data/raw/LA -iname "*eval*" -iname "*.txt"

data/raw/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.female.trl.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.gi.trl.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.female.trn.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.male.trn.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.male.trl.txt
data/raw/LA/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt


In [17]:
!ls data/protocols/ | grep -i eval


ASVspoof2019.LA.cm.eval.trl.txt


In [18]:
!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.eval.trl.txt \
  --n-per-class 750 --output data/protocols/eval_subset.txt

!python -m src.extract_features \
  --protocol data/protocols/eval_subset.txt \
  --audio-dir data/raw/LA/ASVspoof2019_LA_eval/flac \
  --features mfcc lfcc spectral --output data/features/eval.npz

!python -m src.run_experiment \
  --train-features data/features/train.npz \
  --test-features data/features/eval.npz \
  --output-dir results_eval/

import pandas as pd
pd.read_csv('results_eval/comparison.csv')

bonafide: 750/7355 kept
spoof: 750/63882 kept
Wrote 1500 lines -> data/protocols/eval_subset.txt
extracting features: 100% 1500/1500 [00:40<00:00, 36.86it/s]
Saved 1500 utterances -> data/features/eval.npz
  mfcc: dim=120
  lfcc: dim=120
  spectral: dim=12

=== mfcc_svm (mfcc + svm) ===
{'accuracy': 0.8933333333333333, 'precision_spoof': 0.8922872340425532, 'recall_spoof': 0.8946666666666667, 'f1_spoof': 0.8934753661784287, 'roc_auc': 0.9593937777777779, 'eer': 0.10666666666666669, 'eer_threshold': 0.5084365694334596, 'n_samples': 1500, 'features': 'mfcc', 'classifier': 'svm'}

=== lfcc_svm (lfcc + svm) ===
{'accuracy': 0.8753333333333333, 'precision_spoof': 0.9828473413379074, 'recall_spoof': 0.764, 'f1_spoof': 0.859714928732183, 'roc_auc': 0.970359111111111, 'eer': 0.09199999999999998, 'eer_threshold': 0.024473858936983686, 'n_samples': 1500, 'features': 'lfcc', 'classifier': 'svm'}

=== mfcc_rf (mfcc + random_forest) ===
{'accuracy': 0.8613333333333333, 'precision_spoof': 0.89275362

,experiment,features,classifier,accuracy,precision_spoof,recall_spoof,f1_spoof,roc_auc,eer,n_samples
0,mfcc_svm,mfcc,svm,0.893333,0.892287,0.894667,0.893475,0.959394,0.106667,1500
1,lfcc_svm,lfcc,svm,0.875333,0.982847,0.764000,0.859715,0.970359,0.092000,1500
2,mfcc_rf,mfcc,random_forest,0.861333,0.892754,0.821333,0.855556,0.933600,0.142000,1500
3,mfcc_spectral_svm,mfcc+spectral,svm,0.900000,0.900000,0.900000,0.900000,0.962350,0.099333,1500
4,mfcc_lfcc_spectral_svm,mfcc+lfcc+spectral,svm,0.910667,0.988889,0.830667,0.902899,0.983756,0.064667,1500


In [15]:
!python -m src.run_experiment \
  --train-features data/features/train.npz \
  --test-features data/features/dev.npz \
  --output-dir results/

import pandas as pd
pd.read_csv('results/comparison.csv')


=== mfcc_svm (mfcc + svm) ===
{'accuracy': 0.892, 'precision_spoof': 0.8537906137184116, 'recall_spoof': 0.946, 'f1_spoof': 0.8975332068311196, 'roc_auc': 0.9728159999999999, 'eer': 0.08699999999999998, 'eer_threshold': 0.7473035105167128, 'n_samples': 1000, 'features': 'mfcc', 'classifier': 'svm'}

=== lfcc_svm (lfcc + svm) ===
{'accuracy': 0.988, 'precision_spoof': 0.9919354838709677, 'recall_spoof': 0.984, 'f1_spoof': 0.9879518072289156, 'roc_auc': 0.99902, 'eer': 0.015000000000000006, 'eer_threshold': 0.4183168682146115, 'n_samples': 1000, 'features': 'lfcc', 'classifier': 'svm'}

=== mfcc_rf (mfcc + random_forest) ===
{'accuracy': 0.893, 'precision_spoof': 0.9034907597535934, 'recall_spoof': 0.88, 'f1_spoof': 0.8915906788247214, 'roc_auc': 0.9554000000000001, 'eer': 0.11299999999999999, 'eer_threshold': 0.49, 'n_samples': 1000, 'features': 'mfcc', 'classifier': 'random_forest'}

=== mfcc_spectral_svm (mfcc+spectral + svm) ===
{'accuracy': 0.909, 'precision_spoof': 0.8645276292335

,experiment,features,classifier,accuracy,precision_spoof,recall_spoof,f1_spoof,roc_auc,eer,n_samples
0,mfcc_svm,mfcc,svm,0.892,0.853791,0.946,0.897533,0.972816,0.087,1000
1,lfcc_svm,lfcc,svm,0.988,0.991935,0.984,0.987952,0.999020,0.015,1000
2,mfcc_rf,mfcc,random_forest,0.893,0.903491,0.880,0.891591,0.955400,0.113,1000
3,mfcc_spectral_svm,mfcc+spectral,svm,0.909,0.864528,0.970,0.914232,0.979576,0.074,1000
4,mfcc_lfcc_spectral_svm,mfcc+lfcc+spectral,svm,0.995,0.994012,0.996,0.995005,0.999832,0.003,1000


In [19]:
print(pd.read_csv('results_eval/comparison.csv'))


               experiment            features     classifier  accuracy  \
0                mfcc_svm                mfcc            svm  0.893333   
1                lfcc_svm                lfcc            svm  0.875333   
2                 mfcc_rf                mfcc  random_forest  0.861333   
3       mfcc_spectral_svm       mfcc+spectral            svm  0.900000   
4  mfcc_lfcc_spectral_svm  mfcc+lfcc+spectral            svm  0.910667   

   precision_spoof  recall_spoof  f1_spoof   roc_auc       eer  n_samples  
0         0.892287      0.894667  0.893475  0.959394  0.106667       1500  
1         0.982847      0.764000  0.859715  0.970359  0.092000       1500  
2         0.892754      0.821333  0.855556  0.933600  0.142000       1500  
3         0.900000      0.900000  0.900000  0.962350  0.099333       1500  
4         0.988889      0.830667  0.902899  0.983756  0.064667       1500  


In [11]:
!find data/raw -maxdepth 4 -type d
!find data/raw -iname "*.flac" | head -3
!find data/raw -iname "*trn*" -o -iname "*protocol*" | head -10

data/raw
data/raw/LA
data/raw/LA/ASVspoof2019_LA_eval
data/raw/LA/ASVspoof2019_LA_eval/flac
data/raw/LA/ASVspoof2019_LA_cm_protocols
data/raw/LA/ASVspoof2019_LA_asv_protocols
data/raw/LA/ASVspoof2019_LA_dev
data/raw/LA/ASVspoof2019_LA_dev/flac
data/raw/LA/ASVspoof2019_LA_train
data/raw/LA/ASVspoof2019_LA_train/flac
data/raw/LA/ASVspoof2019_LA_asv_scores
data/raw/LA/ASVspoof2019_LA_eval/flac/LA_E_1000147.flac
data/raw/LA/ASVspoof2019_LA_eval/flac/LA_E_1000273.flac
data/raw/LA/ASVspoof2019_LA_eval/flac/LA_E_1000791.flac
data/raw/LA/ASVspoof2019_LA_cm_protocols
data/raw/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.female.trn.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.male.trn.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.female.trn.txt
data/raw/LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.male.trn.t

In [10]:
!pip install -q kaggle
!kaggle datasets download -d anishsarkar22/asvpoof-2019-dataset-la -p data/raw --unzip

Dataset URL: https://www.kaggle.com/datasets/anishsarkar22/asvpoof-2019-dataset-la
License(s): ODC Attribution License (ODC-By)
100% 7.12G/7.12G [01:32<00:00, 82.6MB/s]



In [8]:
!mkdir -p ~/.kaggle && echo KGAT_21804188c7fac8812b6427358bed5a4d > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [3]:
# 2) Install python dependencies
!pip install -q -r requirements.txt

In [6]:
!wget --no-verbose -O data/raw/LA.zip \
  --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36" \
  https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip
!ls -la data/raw/LA.zip
!head -c 300 data/raw/LA.zip

No data received.
No data received.
No data received.
No data received.
No data received.
No data received.
No data received.
^C
-rw-r--r-- 1 root root 0 Sep 13 09:07 data/raw/LA.zip


In [ ]:
# 4) Copy the official protocol files into data/protocols/
!cp data/raw/ASVspoof2019/LA/ASVspoof2019_LA_cm_protocols/*.txt data/protocols/
!ls data/protocols/

In [ ]:
# 5) Subsample to a manageable size (1500/class for train, 500/class for dev)
!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.train.trn.txt \
  --n-per-class 1500 --output data/protocols/train_subset.txt

!python -m src.subsample_protocol \
  --protocol data/protocols/ASVspoof2019.LA.cm.dev.trl.txt \
  --n-per-class 500 --output data/protocols/dev_subset.txt

In [ ]:
# 6) Extract features (MFCC, LFCC, spectral). This is the slowest step — grab a coffee.
!python -m src.extract_features \
  --protocol data/protocols/train_subset.txt \
  --audio-dir data/raw/ASVspoof2019/LA/ASVspoof2019_LA_train/flac \
  --features mfcc lfcc spectral --output data/features/train.npz

!python -m src.extract_features \
  --protocol data/protocols/dev_subset.txt \
  --audio-dir data/raw/ASVspoof2019/LA/ASVspoof2019_LA_dev/flac \
  --features mfcc lfcc spectral --output data/features/dev.npz

In [ ]:
# 7) Run all experiments (MFCC+SVM, LFCC+SVM, MFCC+RF, feature combinations) and compare them
!python -m src.run_experiment \
  --train-features data/features/train.npz \
  --test-features data/features/dev.npz \
  --output-dir results/

import pandas as pd
pd.read_csv('results/comparison.csv')

In [ ]:
# 8) Package everything needed to continue (results, cached features, and the exact subsets used)
# and download it. Send the downloaded spoken_results.zip back in the chat.
!zip -rq spoken_results.zip results data/features data/protocols/train_subset.txt data/protocols/dev_subset.txt

from google.colab import files
files.download('spoken_results.zip')

In [20]:
pd.read_csv('results_eval/mfcc_lfcc_spectral_svm_per_attack.csv')

,attack_type,accuracy,n
0,A18,0.019608,51
1,A17,0.116279,43
2,A12,0.586207,58
3,A15,0.890909,55
4,A10,0.912500,80
5,A14,0.961538,52
6,-,0.990667,750
7,A07,1.000000,66
8,A13,1.000000,59
9,A11,1.000000,72


In [21]:
!zip -rq spoken_results.zip results results_eval data/features data/protocols/*_subset.txt

from google.colab import files
files.download('spoken_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>